# Single Variable Linear Regression


## Data Generation

Pada bagian awal, dimuat tiga library utama yang akan digunakan sepanjang
percobaan regresi variabel tunggal. NumPy berperan sebagai fondasi komputasi
numerik dan manipulasi array, Matplotlib digunakan untuk visualisasi sebaran
data, dan modul random dimanfaatkan untuk membangkitkan bilangan acak pada
pembuatan noise.


In [ ]:
# Memuat lclearibrary yang dibutuhkan dalam pemrosesan data dan pembuatan
# model regresi linier
import numpy as np
import matplotlib.pyplot as plt
import random

In [ ]:
num_samples = 500
# Pendefinisian fungsi yang sebenarnya (fungsi target) y = 4x - 1
true_w = np.array([4.0])
true_b = np.array([-1.0])
# Menghasilkan data input (x-axis)
features = np.random.randn(num_samples)
# Menghasilkan data output (y-axis)
# Catatan: untuk kondisi multivariate -> torch.matmul(features, true_w) + true_b
labels = features * true_w + true_b
# Menambahkan noise pada sebaran data yang ada
labels += np.random.randint(low=1, high=3, size=num_samples) * np.random.randn(
    num_samples
)
# Visualisasi data
plt.figure(figsize=(8, 8))
plt.scatter(features, labels)
plt.show()

Block ini menghasilkan data sintetis berjumlah 500 sampel yang mengikuti pola
linear dengan persamaan:

$$
y = w \cdot x + b
$$

Nilai parameter sebenarnya ditetapkan sebagai `true_w = 4.0` dan
`true_b = -1.0`, yang berfungsi sebagai acuan untuk dibandingkan dengan hasil
estimasi model nantinya. Variabel input x dibangkitkan menggunakan fungsi
`np.random.randn` sehingga memiliki distribusi normal standar.

Agar data tidak terlalu sempurna layaknya data buatan, ditambahkan komponen
noise acak dengan skala antara 1 hingga 2 yang dikalikan dengan distribusi
normal tambahan. Scatter plot yang dihasilkan menjadi alat verifikasi visual
untuk memastikan bahwa pola sebaran data memang mengikuti tren linear sebelum
melangkah ke tahap estimasi parameter.

## Data Preparation

In [ ]:
train_size = 0.7
num_train = int(num_samples * train_size)

# Pembagian data ke training dan testing data
x_train = features[0:num_train]
y_train = labels[0:num_train]
x_test = features[num_train:num_samples]
y_test = labels[num_train:num_samples]

# Visualisasi Data
plt.figure(figsize=(8, 8))
plt.scatter(x_train, y_train, c="Blue")
plt.scatter(x_test, y_test, c="Red")
plt.show()

Dataset keseluruhan dibagi menjadi dua subset dengan perbandingan 70:30,
menghasilkan 350 sampel untuk data latih dan 150 sampel untuk data uji. Proses
pembagian dilakukan melalui slicing array berdasarkan indeks secara langsung.

Visualisasi menggunakan dua warna berbeda (biru untuk data latih dan merah
untuk data uji) membantu memverifikasi bahwa distribusi kedua subset tersebar
secara merata. Hal ini penting untuk memastikan tidak ada bias pemilihan yang
dapat mempengaruhi hasil evaluasi model di kemudian hari.

## Least Square Estimator

In [ ]:
# 1. Menghitung nilai rata-rata (mean) dari x_train dan y_train
x_mean = np.mean(x_train)
y_mean = np.mean(y_train)

# 2. Menghitung gradient (b1) menggunakan rumus turunan Least Square
numerator = np.sum((x_train - x_mean) * (y_train - y_mean))
denominator = np.sum((x_train - x_mean) ** 2)
b1_hat = numerator / denominator

# 3. Menghitung intercept (b0)
b0_hat = y_mean - (b1_hat * x_mean)

In [ ]:
# Evaluasi menggunakan Determination Coefficient (R2). Nilai evaluasi ini berbanding
# terbalik dengan SSR (Sum Square Residual) yang mana semakin besar nilai R2
# (mendekati nilai 1) maka semakin kecil nilai SSR-nya (mendekati nilai 0)
# R2 = 1 - (Sum Square Residual / Total Sum of Squares)
y_train_pred = b0_hat + b1_hat * x_train
ss_res = np.sum((y_train - y_train_pred) ** 2)
ss_tot = np.sum((y_train - y_mean) ** 2)
r_sq = 1 - (ss_res / ss_tot)
print("Coefficient of determination (R2):", r_sq)

# Menampilkan nilai b0 dan b1 yang dihasilkan dari perhitungan LSE
print("Intercept (b0):", b0_hat)
print("Gradient (b1):", b1_hat)

In [ ]:
# Menampilkan garis dari persamaan regresi linear terhadap training data
plt.figure(figsize=(8, 8))
plt.scatter(x_train, y_train, c="Blue")
axes = plt.gca()
x_vals = np.array(axes.get_xlim())

# Menggunakan b0_hat dan b1_hat dari kalkulasi manual
y_vals = b0_hat + b1_hat * x_vals
plt.plot(x_vals, y_vals, "-", c="Green", linewidth=3)
plt.title("Garis Regresi pada Training Data (LSE)")
plt.show()

Block ini menerapkan metode Least Square Estimator (LSE), yaitu pendekatan
closed-form yang menghitung parameter regresi secara langsung tanpa proses
iteratif. Nilai gradient diperoleh melalui persamaan:

$$
\beta_1 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}
$$

Sedangkan intercept dihitung menggunakan:

$$
\beta_0 = \bar{y} - \beta_1 \bar{x}
$$

Dalam implementasi kode, pembilang pada rumus gradient (variabel `numerator`)
merepresentasikan kovarians antara x dan y, sementara `denominator` mewakili
varians dari x. Perhitungan intercept kemudian dilakukan sedemikian rupa agar
garis regresi tepat melintasi titik rata-rata data latih.

Evaluasi kualitas model dilakukan menggunakan koefisien determinasi dengan
rumus:

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
$$

Nilai R² yang mendekati 1 mengindikasikan bahwa sebagian besar varians data
berhasil dijelaskan oleh model. Garis hijau pada plot menunjukkan garis
regresi hasil estimasi, yang secara visual tampak menempel cukup rapat pada
sebaran data latih.

## Testing Using Estimated Regression Parameters

In [ ]:
# Menghitung nilai prediksi dan error (MSE) dari model terhadap testing set
y_test_pred = b0_hat + b1_hat * x_test
mse_test = np.mean((y_test_pred - y_test) ** 2)
print("Mean Squared Error (MSE) pada Testing Data: %.2f" % mse_test)

In [ ]:
# Menampilkan garis dari persamaan regresi linear terhadap testing data
plt.figure(figsize=(8, 8))
plt.scatter(x_test, y_test, c="Red")
axes = plt.gca()
x_vals = np.array(axes.get_xlim())
y_vals = b0_hat + b1_hat * x_vals
plt.plot(x_vals, y_vals, "-", c="Green", linewidth=3)
plt.title("Garis Regresi pada Testing Data")
plt.show()

Parameter $b_0$ dan $b_1$ yang telah diestimasi dari data latih selanjutnya
digunakan untuk melakukan prediksi pada data uji tanpa melalui proses
pelatihan ulang. Kualitas prediksi dievaluasi menggunakan Mean Squared Error:

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2
$$

Nilai MSE yang rendah menandakan kemampuan model dalam melakukan generalisasi
pada data yang belum pernah dilihat sebelumnya. Plot yang menampilkan data uji
(titik merah) beserta garis regresi (hijau) memberikan konfirmasi visual
bahwa garis estimasi tetap relevan untuk subset testing.

# 2. Multivariate Regression dengan ANN

Block kedua memuat library-library untuk implementasi regresi multivariat
menggunakan ANN. Pandas dan Seaborn digunakan untuk eksplorasi dan
visualisasi data, scikit-learn menangani proses pengambilan dataset,
normalisasi, serta pembagian data, sedangkan PyTorch berperan sebagai
framework utama dalam pembangunan dan pelatihan model.

In [ ]:
# Memuat library yang dibutuhkan untuk membangun pemrosesan data dan pembuatan
# model regresi linier multi variable dengan ANN
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from sklearn import preprocessing
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

## Data Exploration

In [ ]:
# Memuat dataset
housing = fetch_california_housing()

# Mengecek tipe dari dataset
print(type(housing))

In [ ]:
# Mengecek informasi yang terdapat dalam dataset
# dictionary keys
print(housing.keys())

In [ ]:
# Mencetak / menampilkan deskripsi dari dataset
print(housing["DESCR"])

In [ ]:
# Catatan yang perlu diperhatikan:
# 1. data = berisi kumpulan data dari feature
# 2. target = berisi kumpulan data dari variable target
# 3. target_names = berisi kumpulan nama dari variable target
# 4. feature_names = berisi kumpulan nama dari feature

# Memuat kumpulan data feature ke dalam pandas DataFrame
housing_df = pd.DataFrame(data=housing["data"], columns=housing["feature_names"])
housing_df

In [ ]:
# Menambahkan informasi varible target pada setiap baris data dalam DataFrame
housing_df[housing["target_names"][0]] = housing["target"]
housing_df.head()

In [ ]:
# Mengecek terhadap nilai yang kosong (missing values)
print(np.sum(np.isnan(housing_df)))

Dataset California Housing dimuat menggunakan fungsi `fetch_california_housing()`
dari scikit-learn. Objek yang dikembalikan bertipe Bunch dengan karakteristik
mirip dictionary, memiliki kunci berupa `data`, `target`, `feature_names`,
`target_names`, dan `DESCR`.

Data feature kemudian dikonversi menjadi DataFrame pandas untuk memudahkan
eksplorasi, disusul dengan penambahan kolom target `MedHouseVal` ke dalam
DataFrame tersebut. Pengecekan missing value menggunakan `np.sum(np.isnan(...))`
menghasilkan nilai 0, yang mengindikasikan bahwa dataset lengkap dan tidak
memerlukan tahap imputasi.

## Feature Correlation Calculation

In [ ]:
# Melakukan perhitungan feature correlation
plt.figure(figsize=(14, 8))
corr = housing_df.corr()
heatmap = sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2g")

In [ ]:
# Negative correlation yang kuat, sama bergunanya dengan positive correlation
# yang kuat. Sehingga perlu dilakukan sort terhadap nilai absotule-nya
abs(corr["MedHouseVal"]).sort_values(ascending=False)

Fungsi `corr()` menghitung korelasi Pearson untuk setiap pasangan kolom dalam
dataset. Hasilnya divisualisasikan sebagai heatmap untuk memudahkan pembacaan,
di mana warna hangat merepresentasikan korelasi positif dan warna dingin
menunjukkan korelasi negatif.

Karena korelasi negatif yang kuat memiliki nilai informatif yang setara dengan
korelasi positif yang kuat, dilakukan pengurutan terhadap nilai absolut
korelasi setiap feature terhadap kolom target `MedHouseVal`. Dari hasil
pengurutan terlihat bahwa `MedInc` menunjukkan hubungan paling kuat dengan
harga rumah, menjadikannya feature yang paling relevan untuk dijadikan masukan
model.

## Data Preparation

In [ ]:
# y merupakan variable dependent, target variable (yang akan diestimasi oleh model kita)
y = housing_df["MedHouseVal"]

# X merupakan features yang akan diberikan kepada model. iloc digunakan untuk
# melakukan slice pada DataFrame, dan diambil feature dengan corr yang baik dengan target
X = housing_df.loc[:, ("MedInc", "AveRooms", "Latitude", "HouseAge")]

# Menampilkan ukuran dari setiap variabel X dan y
print(f"Data X = {X.shape}")
print(f"Data y = {y.shape}")

In [ ]:
# Melakukan normalisasi data ke dalam rentang nilai data yang seragam
min_max_scaler = preprocessing.MinMaxScaler()
X = min_max_scaler.fit_transform(X.values.reshape(-1, X.shape[1]))
y = min_max_scaler.fit_transform(y.values.reshape(-1, 1))

X = pd.DataFrame(X)
y = pd.DataFrame(y)

print(f"Contoh data X setelah normalisasi \n{X.head()}\n")
print(f"Contoh data y setelah normalisasi \n{y.head()}")

In [ ]:
# Proses pembagian kumpulan data ke dalam data training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

Dipilih empat feature dengan korelasi tertinggi terhadap target, yaitu
`MedInc`, `AveRooms`, `Latitude`, dan `HouseAge`. Sebelum masuk ke tahap
pelatihan, seluruh feature dan target dinormalisasi ke rentang [0, 1]
menggunakan MinMaxScaler dengan persamaan:

$$
X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}
$$

Tahap normalisasi menjadi krusial mengingat skala antar feature sangat
bervariasi, sementara arsitektur jaringan saraf cukup sensitif terhadap
perbedaan skala masukan. Setelah normalisasi, data dibagi menjadi subset latih
dan uji dengan rasio 70:30 menggunakan `train_test_split`, di mana parameter
`random_state=0` ditetapkan untuk memastikan hasil pembagian dapat
direproduksi.

## Model Definition

In [ ]:
# Pembentukan class dari arsitektur model sebagai tujuan proses training
class LinearRegressor(torch.nn.Module):
    def __init__(self, n_features, n_outputs):
        super().__init__()
        self.net = torch.nn.Linear(n_features, n_outputs)

    def forward(self, x):
        return self.net(x)

# Beberapa catatan penting:
# 1. super().__init__() digunakan sebagai metode untuk melakukan inisialisasi variabel dari parent class
# 2. n_feature berisi jumlah total variabel dari feature
# 3. n_output berisi jumlah variable target yang harus diestimasi
# 4. Kita akan mengubah penggunaan y = linreg(data) menjadi y = LinearRegressor.forward(data)

In [ ]:
# Penggunaan arsitektur yang akan dilakukan pelatihan dan pengujian
n_features = X.shape[1]  # 4
n_outputs = 1

model = LinearRegressor(n_features, n_outputs)
print(model)

In [ ]:
# Menampilkan nama dan nilai dari semua trainable parameter
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.data)

Model didefinisikan sebagai class `LinearRegressor` yang mewarisi
`torch.nn.Module`. Arsitektur model terdiri dari satu layer `torch.nn.Linear`
yang memetakan 4 feature masukan menjadi 1 keluaran, sehingga secara matematis
ekuivalen dengan persamaan regresi linear multivariat:

$$
\hat{y} = w_0 x_0 + w_1 x_1 + w_2 x_2 + w_3 x_3 + b
$$

Method `forward` menentukan alur komputasi yang dieksekusi setiap kali model
dipanggil. Melalui pencetakan struktur model dan perulangan
`named_parameters()`, dapat diamati nilai awal bobot dan bias yang masih
diinisialisasi secara acak oleh PyTorch sebelum proses pelatihan dimulai.

## Training Process

In [ ]:
# Inisialisasi beberapa parameter dan fungsi untuk mendukung proses training
lr = 0.001
num_epochs = 100
batch_size = 256
optimizer = torch.optim.SGD(model.parameters(), lr)
loss_function = torch.nn.MSELoss()

In [ ]:
# Konversi DataFrame ke dalam torch tensors
training_set = torch.utils.data.TensorDataset(
    torch.tensor(X_train.values, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32))
test_set = torch.utils.data.TensorDataset(
    torch.tensor(X_test.values, dtype=torch.float32),
    torch.tensor(y_test.values, dtype=torch.float32))

# Pembuatan DataLoader untuk kedua kumpulan datam, yang melibatkan
# sampling batches khususnya untuk data training
training_loader = torch.utils.data.DataLoader(training_set, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, shuffle=False)

In [ ]:
# Mengatur mode training untuk model
model.train()

loss_train = []
for epoch in range(num_epochs):
    loss_batch = []
    for input, target in training_loader:
        # Optimizer gradient diatur menjadi zero, sehingga dapat dihitung ulang untuk langkah pelatihan terte
        optimizer.zero_grad()
    
        # Model memberikan luaran regresi
        output = model(input)
        
        # Perhitungan loss function
        loss = loss_function(output, target)
        loss_batch.append(loss.detach().cpu())
        
        # Menghitung gradien loss sehubungan dengan setiap parameter model yang akan diperbarui.
        # Ini dikenal sebagai backpropagation
        loss.backward()
        
        # Memperbaharui parameter sehubungan dengan gradien loss untuk memperbaikinya
        optimizer.step()

    loss_train.append((np.mean(loss_batch)))
    print(f"[Epoch {epoch + 1}] - Training loss: {loss_train[-1]}")

In [ ]:
# Melakukan plot kualitas model (loss) vs training epochs
plt.figure(figsize=(10, 5))
plt.plot(loss_train, '-', color='blue', label='Train Loss')
plt.title('Loss over epochs')
plt.xlabel('Epochs')

plt.ylabel('Loss')
plt.show()

Proses pelatihan dikonfigurasi dengan optimizer SGD, learning rate 0.001,
jumlah epoch 100, dan batch size 256. Data latih dikemas dalam `TensorDataset`
kemudian dimuat ke `DataLoader` agar dapat diproses per batch dengan urutan
yang diacak pada setiap epoch.

Setiap iterasi pelatihan mengikuti lima langkah fundamental dalam siklus
pembelajaran neural network, yaitu pengosongan gradient melalui
`optimizer.zero_grad()`, eksekusi forward pass untuk menghasilkan prediksi,
perhitungan loss menggunakan MSE, backpropagation melalui `loss.backward()`
untuk menyebarkan error, dan pembaruan bobot melalui `optimizer.step()`.
Secara matematis, pembaruan parameter mengikuti aturan:

$$
\theta_{baru} = \theta_{lama} - \eta \cdot \nabla_\theta L
$$

Nilai loss rata-rata pada setiap epoch dicatat dan divisualisasikan dalam
bentuk kurva. Tren loss yang menurun dan cenderung mendatar menjadi indikator
bahwa model sedang dalam proses pembelajaran dari data latih.

## Testing Process

In [ ]:
# Mengatuh mode testing untuk model
model.eval()

test_losses = []
for input, target in test_loader:
    # Melakukan non aktif dari gradient computation
    with torch.no_grad():
        output = model(input) # model.forward(input)
        loss = loss_function(output, target)
        test_losses += [loss.item()]
        
print(f"Mean test loss = {np.mean(test_losses)}")

In [ ]:
print(f"Nilai dari bobot(w): \n{model.net.weight}\n")
print(f"Nilai dari bias(b): \n{model.net.bias}")

In [ ]:
all_predicted_values = []

# Membuat prediksi menggunakan model yang telah dilatih dan menyimpan hasilnya
with torch.no_grad():
    for input_data, _ in test_loader:
        predicted_values_batch = model(input_data)
        all_predicted_values.append(predicted_values_batch.cpu().numpy())
        
predicted_values = np.concatenate(all_predicted_values, axis=0).reshape(-1, 1)

In [ ]:
# Melakukan scatter plot dari actual vs predicted values
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test.values.flatten(), y=predicted_values.flatten())
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Linear Regression - Actual vs Predicted Values')
plt.show()

Pada tahap pengujian, model dialihkan ke mode `eval()` dan komputasi gradient
dinonaktifkan melalui `torch.no_grad()` untuk mempercepat proses inference
serta menghemat penggunaan memori. Loss rata-rata pada data uji dicetak,
diikuti dengan tampilan nilai bobot dan bias akhir hasil pelatihan.

Scatter plot yang membandingkan nilai aktual dengan nilai prediksi
menunjukkan titik-titik yang berkumpul di sekitar garis diagonal,
mengindikasikan prediksi yang mendekati nilai sebenarnya. Meskipun demikian,
masih terlihat sebaran yang cukup lebar karena hubungan antara feature dengan
harga rumah pada kenyataannya tidak sepenuhnya linear.

## Kesimpulan

Berdasarkan eksplorasi pada modul ini, regresi linear dapat diimplementasikan melalui dua pendekatan utama. Pendekatan pertama adalah Least Square Estimator (LSE) yang memberikan solusi closed-form secara eksak. Metode ini sangat efisien untuk kasus univariat, namun memiliki keterbatasan komputasi jika diskalakan ke dataset berukuran besar. Pendekatan kedua memanfaatkan Artificial Neural Network (ANN) dengan optimizer Stochastic Gradient Descent (SGD) yang bekerja secara iteratif melalui proses backpropagation. 

Meskipun keduanya sama-sama bertujuan meminimalkan error kuadrat (MSE), pendekatan ANN terbukti lebih fleksibel untuk menangani regresi multivariat (seperti prediksi harga rumah pada dataset California Housing). Dalam implementasi ANN, tahapan praproses seperti normalisasi Min-Max dan penentuan hyperparameter (learning rate, batch size, dan epoch) memegang peranan krusial dalam memastikan model konvergen dengan baik tanpa mengalami overfitting. Pemahaman ini menjadi fondasi penting untuk pengembangan arsitektur deep learning yang lebih kompleks pada eksperimen selanjutnya.